<img src="https://datasciencedegree.wisconsin.edu/wp-content/themes/data-gulp/images/logo.svg" width="300">

 # <center>DS 710 - Lesson 6a</center>

This notebook will work through some of the basics of the pandas library.

References for the material in this notebook include 
* Chapters 5-8 of your text (the author of the book is also one of the authors of pandas, and it shows!), 
* Official pandas Documentation including the <a href="https://pandas.pydata.org/docs/user_guide/index.html">User Guide</a> and the <a href="https://pandas.pydata.org/docs/reference/index.html#api">API Reference</a>.

---

### Getting started

We'll start by importing NumPy and pandas:

In [ ]:
import numpy as np
import pandas as pd

Next, we'll read some data from a csv. Make sure you have downloaded `elements-by-episode.csv` and have it in the same directory as this notebook (Source: <a href="https://github.com/fivethirtyeight/data">FiveThirtyEight</a>). The data given indicates what Bob Ross painted in each episode of "The Joy of Painting", which aired new episodes on PBS from 1983 to 1994. A "1" in an entry indicates that Ross painted the item tagged in that episode, while a "0" indicates that the tagged object was not present in an episode. Note that one column indiactes whether it was his son, Steve Ross, who was painting in that episode instead. The `head` function will show us the first five rows of our DataFrame.

In [ ]:
bob_ross = pd.read_csv('elements-by-episode.csv')
bob_ross.head()

We can see that this is a pandas dataframe, find the number of rows and columns in the dataframe, and see the names of all columns as follows:

In [ ]:
print(type(bob_ross))
print(bob_ross.shape)
print(bob_ross.columns)

We can determine the datatypes of the entries of each column using `dtypes`:

In [ ]:
bob_ross.dtypes

We can pull out the column `TREE` in several ways (Ross painted a lot of trees).

In [ ]:
bob_ross.TREE

In [ ]:
bob_ross['TREE']

---

### Filtering the data

There are quite a few columns in the dataframe (69, in fact); we can choose a few particular columns to display. Suppose we want to know about the presence of trees and mountains.

In [ ]:
bob_ross_tm = bob_ross[['EPISODE', 'TITLE', 'CONIFER', 'DECIDUOUS', 'PALM_TREES', 'MOUNTAIN', 'MOUNTAINS', 'SNOWY_MOUNTAIN', 'TREE', 'TREES']]
bob_ross_tm.head(10)

Maybe we only care about Season 15 of "The Joy of Painting". Then, we can select just the rows that correspond to Season 15, as follows.

In [ ]:
bob_ross_tm_S15 = bob_ross_tm[bob_ross_tm.EPISODE.str.contains('S15')]
bob_ross_tm_S15

🧠 In the cell below, try writing code to display all episodes from the original file whose titles contain the word "SNOW".

The `mean` function will calculate the means of columns; starting with `pandas` 2.0, you need to specify `numeric_only=True` if you want to ignore columns with non-numeric values.

🧠 Which category of tree/mountain was most popular in Season 15? Did Ross ever paint a single tree in a painting during Season 15?

In [ ]:
bob_ross_tm_S15.mean(numeric_only = True)

---

### Adding a new column, & views/copies

Let's add a column that is the sum of all the tree type columns: that is, a column that will tell us how many of the three specific tree types appear in each painting.

#### A first attempt to add a new trees column

First, we want to offer you a training experience on writing correct pandas code.  So, we'll start with some code that doesn't work (in that it generates a warning).

⚠️ We would really like to be able to write the following code:

In [ ]:
# this code will generate a warning
bob_ross_tm = bob_ross[['EPISODE', 'TITLE', 'CONIFER', 'DECIDUOUS', 'PALM_TREES', 'MOUNTAIN', 'MOUNTAINS', 'SNOWY_MOUNTAIN', 'TREE', 'TREES']]
bob_ross_tm['TREE_TYPE_TOTALS'] = bob_ross_tm.CONIFER + bob_ross_tm.DECIDUOUS + bob_ross_tm.PALM_TREES

In [ ]:
# this code will generate the same warning, the first time it is run
bob_ross = pd.read_csv('elements-by-episode.csv')
bob_ross_tm = bob_ross[['EPISODE', 'TITLE', 'CONIFER', 'DECIDUOUS', 'PALM_TREES', 'MOUNTAIN', 'MOUNTAINS', 'SNOWY_MOUNTAIN', 'TREE', 'TREES']]
bob_ross_tm.loc['TREE_TYPE_TOTALS'] = bob_ross_tm.CONIFER +  bob_ross_tm.DECIDUOUS +  bob_ross_tm.PALM_TREES

##### Why does this happen?

Do your best to [grok](https://en.wikipedia.org/wiki/Grok) this.  It may take some time for this to make sense, and that's ok!

Short answer: views.

* An SO post, [How to deal with SettingWithCopyWarning in Pandas](https://stackoverflow.com/questions/20625582/how-to-deal-with-settingwithcopywarning-in-pandas), discusses the warning, and how to handle it.  In particular, see the answer by user cs95, section "The "XY Problem": What am I doing wrong?".  
* Here's some official documentation about the topic. [Pandas documentation: Indexing and selecting data](https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html).  
* And this post, [Real Python: SettingWithCopyWarning in Pandas: Views vs Copies](https://realpython.com/pandas-settingwithcopywarning/) has a great exposition of the problem, complete with pictures!  (Use an ad-blocker, or the page is unviewable 😕)

Summary of why the error happens:
* Many operations in pandas produce "views" instead of copies.  This is for performance, so that tons of data isn't needlessly copied in memory.  Views are containers that "point to" existing data, instead of duplicating it.
* Other operations in pandas produce "copies" -- so that the data is duplicated.
* Setting data in views can produce weird behaviour, because the setting operation can modify a container you didn't expect to modify.

The specific cause in this case:
* When we made `bob_ross_tm`, we produced a view into the old data frame.  So, setting data into this data frame may produce the warning.

A specific solution in this case:
* When making `bob_ross_tm`, make it a *copy* of the data.  
* That is, use `bob_ross_tm = bob_ross[['EPISODE']].copy()

There are other solutions, too. See below for some options.  

##### How to tell if you have copy /view

It's not immediately obvious how to tell whether you have a copy or a view into another data frame / object in pandas.  This SO post, [Checking whether data frame is copy or view in Pandas](https://stackoverflow.com/questions/26879073/checking-whether-data-frame-is-copy-or-view-in-pandas), gives some instruction.  

##### Conclusion
Listen to this warning.  Practice will lead you to program using pandas in ways that don't produce this warning.  And, don't give in to the temptation to disable the warning.  That way lies the dark side (and nasty bugs 🦂).

#### More-correct ways of adding the new column

These are several ways of adding the new `TREE_TYPE_TOTALS` column.  All of them essentially copy the data, solving the view/copy problem.  See the next three cells.  (Note, we have the import and read_csv calls so that it makes it easy to play with these cells, if you restart the kernel to get a fresh start).

In [ ]:
# solution 1 -- use .assign
# https://pandas.pydata.org/pandas-docs/version/0.23.1/generated/pandas.DataFrame.assign.html
# produces a deep copy, getting around the view problem
import pandas as pd
bob_ross = pd.read_csv('elements-by-episode.csv')
bob_ross_tm = bob_ross[['EPISODE', 'TITLE', 'CONIFER', 
                        'DECIDUOUS', 'PALM_TREES', 'MOUNTAIN', 
                        'MOUNTAINS', 'SNOWY_MOUNTAIN', 'TREE', 'TREES']]
bob_ross_tm = bob_ross_tm.assign(TREE_TYPE_TOTALS =  bob_ross_tm.CONIFER +  bob_ross_tm.DECIDUOUS +  bob_ross_tm.PALM_TREES)
bob_ross_tm

In [ ]:
# solution 2a -- add `.copy()` when making `bob_ross_tm`
# then use `loc` or whatever other method you want
import pandas as pd
bob_ross = pd.read_csv('elements-by-episode.csv')
bob_ross_tm = bob_ross[['EPISODE', 'TITLE', 'CONIFER', 
                        'DECIDUOUS', 'PALM_TREES', 'MOUNTAIN', 
                        'MOUNTAINS', 'SNOWY_MOUNTAIN', 'TREE', 'TREES']].copy()
bob_ross_tm.loc[:,'TREE_TYPE_TOTALS'] = bob_ross_tm.CONIFER +  bob_ross_tm.DECIDUOUS +  bob_ross_tm.PALM_TREES

In [ ]:
# solution 2b -- add `.copy()` when making `bob_ross_tm`
# then we can use the line we really wanted to the first time
import pandas as pd
bob_ross = pd.read_csv('elements-by-episode.csv')
bob_ross_tm = bob_ross[['EPISODE', 'TITLE', 'CONIFER', 
                        'DECIDUOUS', 'PALM_TREES', 'MOUNTAIN', 
                        'MOUNTAINS', 'SNOWY_MOUNTAIN', 'TREE', 'TREES']].copy()
bob_ross_tm['TREE_TYPE_TOTALS'] = bob_ross_tm.CONIFER + bob_ross_tm.DECIDUOUS + bob_ross_tm.PALM_TREES

---

### Sorting 

We can order the episodes according to how many distinct tree types appear, and then by episode number.

In [ ]:
bob_ross_tm = bob_ross_tm.sort_values(by=['TREE_TYPE_TOTALS','EPISODE'])
bob_ross_tm

Suppose we wish to determine the number of episodes in which each of the tree and mountain categories appears. We can sum down across rows (down columns) as follows:

In [ ]:
bob_ross_tm.sum(axis=0)

Finally, let's write our manipulated data back to a new csv:

In [ ]:
bob_ross_tm.to_csv('bob-ross_trees_mountains.csv', index=False)

🧠 Play with other sorts

In [ ]:
# your code here


---

## When you're done

🧠 After completing the guided practice above, log onto Canvas to complete Quiz 6a. You may wish to keep this notebook open while you work.